# Microsoft Agent Framework Demo
Please visit https://aka.ms/agentframework for the official code samples

In [23]:
from agent_framework import ChatAgent, AgentProtocol, AgentThread, HostedMCPTool
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import AzureCliCredential
from typing import Any

In [24]:
async def main():
    async with (
        AzureCliCredential() as credential,
        ChatAgent(
            chat_client=AzureAIAgentClient(async_credential=credential),
            instructions="You are good at telling jokes."
        ) as agent,
    ):
        result = await agent.run("Tell me a joke about a pirate.")
        print(result.text)

await main()

Why did the pirate go to school?

To improve his "arrr-ticulation"!


In [25]:
"""
Azure AI Agent with Hosted MCP Example

This sample demonstrates integration of Azure AI Agents with hosted Model Context Protocol (MCP)
servers, including user approval workflows for function call security.
"""

async def handle_approvals_with_thread(query: str, agent: "AgentProtocol", thread: "AgentThread"):
    """Here we let the thread deal with the previous responses, and we just rerun with the approval."""
    from agent_framework import ChatMessage

    result = await agent.run(query, thread=thread, store=True)
    while len(result.user_input_requests) > 0:
        new_input: list[Any] = []
        for user_input_needed in result.user_input_requests:
            print(
                f"User Input Request for function from {agent.name}: {user_input_needed.function_call.name}"
                f" with arguments: {user_input_needed.function_call.arguments}"
            )
            new_input.append(
                ChatMessage(
                    role="user",
                    contents=[user_input_needed.create_response(True)],
                ),
            )
        result = await agent.run(new_input, thread=thread, store=True)
    return result


async def main() -> None:
    """Example showing Hosted MCP tools for a Azure AI Agent."""
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as chat_client,
    ):
        # enable azure-ai observability
        await chat_client.setup_azure_ai_observability()
        agent = chat_client.create_agent(
            name="DocsAgent",
            instructions="You are a helpful assistant that can help with microsoft documentation questions.",
            tools=HostedMCPTool(
                name="Microsoft Learn MCP",
                url="https://learn.microsoft.com/api/mcp",
            ),
        )
        thread = agent.get_new_thread()
        # First query
        query1 = "How do I create an Azure storage account using az cli?"
        print(f"User: {query1}")
        result1 = await handle_approvals_with_thread(query1, agent, thread)
        print(f"{agent.name}: {result1}\n")
        print("\n=======================================\n")
        # Second query
        query2 = "What is Microsoft Semantic Kernel?"
        print(f"User: {query2}")
        result2 = await handle_approvals_with_thread(query2, agent, thread)
        print(f"{agent.name}: {result2}\n")

await main()

[2025-10-01 16:49:04 - c:\Users\gugregor\AppData\Local\Programs\Python\Python313\Lib\site-packages\agent_framework_azure_ai\_chat_client.py:223 - WARNING] No Application Insights connection string found for the Azure AI Project, please call setup_observability() manually.


User: How do I create an Azure storage account using az cli?
User Input Request for function from DocsAgent: microsoft_docs_search with arguments: {"query":"create Azure storage account using az cli"}
User Input Request for function from DocsAgent: microsoft_docs_search with arguments: {"query":"create Azure storage account using az cli"}
DocsAgent: You can create an Azure storage account using the Azure CLI (az cli) with the az storage account create command. Below are the steps and a sample command:

1. Make sure you are signed in with az login.
2. If you don't have a resource group, create one using:
   ```azurecli
   az group create --name <resource-group-name> --location <location>
   ```

3. Create a Standard general-purpose v2 storage account with this command:
   ```azurecli
   az storage account create \
     --name <account-name> \
     --resource-group <resource-group-name> \
     --location <location> \
     --sku Standard_RAGRS \
     --kind StorageV2 \
     --min-tls-vers